# 시퀀스 질문 검증

두 단계로 확인합니다. 먼저 DB 없이 가상 이벤트로 질문 분류·청크 간 정렬·근거 연결·빈 결과를 재현합니다. 이후 실제 영상이 처리된 환경에서 `/query`를 호출해 원본 영상과 답변을 대조합니다. 가상 이벤트 통과는 영상 분석 정확도를 증명하지 않습니다.

In [ ]:
import csv
import json
import sys
from pathlib import Path
from urllib import request, error

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
from pipeline.search.sequence_query import is_sequence_query, run_sequence_query
print('프로젝트:', project_root)

## 1. 로컬 로직 검사

입력 리스트를 일부러 뒤섞었습니다. 결과가 `바닥 → 소파 → 내려옴 → 재등장` 순서이고 각 항목에 원본 이벤트 ID와 재생 시각이 붙는지 확인합니다.

In [ ]:
def sample_event(event_id, chunk, second, summary):
    return {'id': event_id, 'video_id': 'sequence-smoke', 'start_time': second,
            'end_time': second + 3, 'summary': summary, 'confidence': 0.9,
            'source_frames': [{'frame_id': f'frame-{event_id}',
                's3_key': f'frames/sequence-smoke/petcam_20261104_090000_{chunk:03d}_frame_{second}.jpg',
                'timestamp': second}]}

events = [sample_event(4, 1, 8, '강아지가 다시 나타남'),
          sample_event(2, 0, 20, '강아지가 소파 위에 있음'),
          sample_event(1, 0, 5, '강아지가 바닥을 달림'),
          sample_event(3, 1, 2, '강아지가 소파에서 내려옴')]
question = '강아지 행동을 시간순으로 알려줘'
assert is_sequence_query(question)
result = run_sequence_query(question, video_id='sequence-smoke', top_k=10, events=events)
assert [e['id'] for e in result['behavior_events']] == [1, 2, 3, 4]
assert [e['source_event_id'] for e in result['evidence_items']] == [1, 2, 3, 4]
assert result['total_event_count'] == 4
print(result['answer'])
print('근거 이벤트 ID:', [e['source_event_id'] for e in result['evidence_items']])
empty = run_sequence_query(question, recording_date='2026-11-05', events=events)
assert empty['total_event_count'] == 0 and not empty['evidence_items']
print('빈 날짜:', empty['answer'])

## 2. 실제 영상의 잠정 관찰 목록

아래 CSV는 공개 영상을 몇 시점에서 직접 확인해 적은 **잠정 평가 자료**입니다. 초 단위 경계는 전체 영상을 연속 재생하며 다시 확정해야 합니다. 서비스에 영상을 넣고 처리하기 전에는 실제 정확도를 계산할 수 없습니다.

In [ ]:
with (project_root / 'docs/sequence-pilot-intervals.csv').open(encoding='utf-8', newline='') as f:
    reference = list(csv.DictReader(f))
core_ids = ['smartbench_0009', 'smartbench_0019', 'smartbench_0041', 'smartbench_0065']
for video in core_ids:
    rows = [r for r in reference if r['video_id'] == video]
    print(video, rows[0]['source_url'])
    for r in rows:
        print(' ', r['event_order'], r['start_s'] + '~' + r['end_s'] + '초', r['action'])

## 3. 처리된 영상의 API 결과 확인

API와 DB에 실제로 처리된 영상의 ID를 아래에 입력합니다. SmartHome-Bench의 `smartbench_0065` 같은 ID는 로컬 파일명과 실제 DB의 `video_id`가 일치할 때만 사용합니다. `top_k=10`으로 요청해 화면 기본값 3개 때문에 뒤의 행동이 잘리지 않도록 합니다. API가 켜져 있지 않거나 ID를 비워 두면 이 셀은 실행을 건너뜁니다.

In [ ]:
api_base_url = 'http://127.0.0.1:8000'
video_id = ''  # 실제 처리된 영상 ID로 변경
recording_date = ''  # 필요하면 YYYY-MM-DD
questions = {'smartbench_0009': '다람쥐와 고양이가 등장한 순서를 알려줘',
             'smartbench_0019': '강아지가 현관 앞에서 움직인 순서를 알려줘',
             'smartbench_0041': '고양이가 도어벨 앞에서 한 행동 순서를 알려줘',
             'smartbench_0065': '강아지가 소파에 있었다가 다시 나타난 순서를 알려줘'}
question = questions.get(video_id, '이 영상의 행동 순서를 알려줘')

if not video_id:
    print('영상 처리 후 video_id를 설정하고 다시 실행하세요.')
else:
    payload = {'query': question, 'video_id': video_id, 'top_k': 10}
    if recording_date:
        payload['recording_date'] = recording_date
    req = request.Request(api_base_url.rstrip('/') + '/query',
                          data=json.dumps(payload).encode('utf-8'),
                          headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with request.urlopen(req, timeout=30) as response:
            actual = json.load(response)
        print('유형:', actual.get('query_type'), '전체 이벤트:', actual.get('total_event_count'),
              '표시:', len(actual.get('results', [])))
        print(actual['answer'])
        for row in actual.get('results', []):
            print('근거:', row.get('source_event_id'), row.get('timestamp'), row.get('s3_key'))
    except (error.URLError, TimeoutError) as exc:
        print('API 호출 실패:', exc)

## 판정표

영상별로 (1) 행동 추출 누락, (2) 출력 순서 역전, (3) 행동 설명 오류, (4) 근거 재생 위치 오류를 각각 기록합니다. `total_event_count`가 충분해도 질문에 필요한 행동이 빠질 수 있으므로 숫자만으로 통과시키지 않습니다. 특정 행동 *다음에 무엇을 했는지* 묻는 질문은 현재 전체 타임라인을 반환하므로 아직 통과 판정 대상이 아닙니다.